In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score


In [20]:
df = pd.read_csv('../Data/final_dataset.csv')

In [21]:
y = df['target']
x = df.drop('target', axis=1) 

In [22]:
train = df[df['datetime'] < '2015-06-01']
test  = df[df['datetime'] >= '2015-06-01']

drop_cols = ['target', 'failure_flag', 'datetime', 'last_maint_datetime']

X_train = train.drop(columns=drop_cols)
y_train = train['target']

X_test = test.drop(columns=drop_cols)
y_test = test['target']

In [23]:
cat_features = ['comp', 'model']
num_features = [col for col in X_train.columns if col not in cat_features]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', 'passthrough', num_features)
    ]
)

In [24]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 47.06781013163143


In [25]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', xgb)
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    504718
           1       0.60      0.97      0.74      9606

    accuracy                           0.99    514324
   macro avg       0.80      0.98      0.87    514324
weighted avg       0.99      0.99      0.99    514324

[[498426   6292]
 [   324   9282]]
